# Lecture 19: Stability of Least Squares Algorithms
## Trefethen & Bau — Numerical Linear Algebra

Python demonstrations using NumPy and Matplotlib.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## CGS, MGS, and Householder Implementations

In [ ]:
def modified_gs(A):
    """Modified Gram-Schmidt QR."""
    A = A.copy().astype(float)
    m, n = A.shape
    Q = np.zeros((m, n))
    R = np.zeros((n, n))
    for j in range(n):
        R[j, j] = np.linalg.norm(A[:, j])
        Q[:, j] = A[:, j] / R[j, j]
        for k in range(j + 1, n):
            R[j, k] = Q[:, j] @ A[:, k]
            A[:, k] -= R[j, k] * Q[:, j]
    return Q, R

## The T&B Experiment: Fitting $e^{\sin(4t)}$ by Degree-14 Polynomial

100 equally spaced points on $[0, 1]$, fit with a polynomial of degree 14 (15 unknowns).  
The solution vector is normalized so $x_{15} = 1$ in exact arithmetic.

In [ ]:
m, n = 100, 15
t = np.arange(m) / (m - 1)
A = np.column_stack([t**j for j in range(n)])
b = np.exp(np.sin(4 * t))

# Get normalization constant so x[14] = 1 (approximately)
x_ref = np.linalg.lstsq(A, b, rcond=None)[0]
norm_const = x_ref[-1]
print(f"Normalization constant: {norm_const:.15f}")

b = b / norm_const
x_ref = np.linalg.lstsq(A, b, rcond=None)[0]
y = A @ x_ref

# Residual plot
plt.figure(figsize=(8, 3.5))
plt.plot(t, b - y, 'b-', linewidth=1)
plt.title('Residual of degree-14 least squares fit to $e^{\\sin(4t)}$')
plt.xlabel('$t$')
plt.ylabel('$b - Ax$')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"||residual|| / ||b|| = {np.linalg.norm(b - y) / np.linalg.norm(b):.2e}")

### Conditioning Parameters (Theorem 18.1)

In [ ]:
kappa = np.linalg.cond(A)
theta = np.arcsin(np.linalg.norm(y - b) / np.linalg.norm(b))
eta = np.linalg.norm(A) * np.linalg.norm(x_ref) / np.linalg.norm(y)

print(f"kappa(A) = {kappa:.4e}")
print(f"theta    = {theta:.4e}  (tiny residual angle)")
print(f"eta      = {eta:.4e}")
print(f"\nEffective condition number for x:")
print(f"  kappa + kappa^2 * tan(theta) / eta = {kappa + kappa**2 * np.tan(theta) / eta:.4e}")
print(f"  (explains the ~{16 - np.log10(kappa + kappa**2*np.tan(theta)/eta):.0f} correct digits we'll see)")

### Compare All Methods

In [ ]:
# 1. Householder QR (NumPy's lstsq / backslash)
x_lstsq = np.linalg.lstsq(A, b, rcond=None)[0]

# 2. Householder QR (explicit)
Q_hh, R_hh = np.linalg.qr(A)
x_hh = np.linalg.solve(R_hh, Q_hh.T @ b)

# 3. MGS (direct Q^T b — loses accuracy due to non-orthogonal Q)
Q_mgs, R_mgs = modified_gs(A)
x_mgs_direct = np.linalg.solve(R_mgs, Q_mgs.T @ b)

# 4. MGS (augmented matrix trick — much better)
Ab = np.column_stack([A, b])
Q_aug, R_aug = modified_gs(Ab)
x_mgs_aug = np.linalg.solve(R_aug[:n, :n], R_aug[:n, n])

# 5. Normal equations
x_ne = np.linalg.solve(A.T @ A, A.T @ b)

# 6. SVD
U, S, Vt = np.linalg.svd(A, full_matrices=False)
x_svd = Vt.T @ ((U.T @ b) / S)

print(f"{'Method':<30s} {'x[14]':>20s} {'|x[14] - 1|':>15s}")
print('-' * 68)
for name, x in [
    ('lstsq (backslash)', x_lstsq),
    ('Householder QR', x_hh),
    ('MGS (direct Q^Tb)', x_mgs_direct),
    ('MGS (augmented)', x_mgs_aug),
    ('Normal equations', x_ne),
    ('SVD', x_svd),
]:
    print(f"{name:<30s} {x[-1]:20.15f} {abs(x[-1]-1):15.2e}")

In [ ]:
# Orthogonality of Q from MGS vs Householder
print(f"||Q^TQ - I|| (Householder): {np.linalg.norm(Q_hh.T @ Q_hh - np.eye(n)):.2e}")
print(f"||Q^TQ - I|| (MGS):         {np.linalg.norm(Q_mgs.T @ Q_mgs - np.eye(n)):.2e}")
print(f"\nMGS's non-orthogonal Q is why direct Q^Tb fails.")
print(f"The augmented matrix trick avoids using Q explicitly.")

## Normal Equations Disaster: Controlled Experiment

Build a matrix with prescribed $\kappa(A)$ and compare QR vs. normal equations.

In [ ]:
np.random.seed(0)
m, n = 100, 10

U, _ = np.linalg.qr(np.random.randn(m, n))
V, _ = np.linalg.qr(np.random.randn(n, n))
s = np.logspace(0, -8, n)  # kappa = 1e8
A = U @ np.diag(s) @ V.T

x_true = np.random.randn(n)
b = A @ x_true  # small residual (theta ≈ 0)

# Householder QR
Q, R = np.linalg.qr(A)
x_qr = np.linalg.solve(R, Q.T @ b)

# Normal equations
x_ne = np.linalg.solve(A.T @ A, A.T @ b)

# SVD
x_svd = np.linalg.lstsq(A, b, rcond=None)[0]

print(f"kappa(A)    = {np.linalg.cond(A):.2e}")
print(f"kappa(A^TA) = {np.linalg.cond(A.T @ A):.2e}")
print(f"\nRelative errors (small residual):")
print(f"  QR:     {np.linalg.norm(x_qr - x_true)/np.linalg.norm(x_true):.2e}")
print(f"  NE:     {np.linalg.norm(x_ne - x_true)/np.linalg.norm(x_true):.2e}")
print(f"  SVD:    {np.linalg.norm(x_svd - x_true)/np.linalg.norm(x_true):.2e}")

## Sweep: Accuracy vs. $\kappa(A)$ for All Three Methods

In [ ]:
np.random.seed(42)
m, n = 80, 10
kappas = np.logspace(1, 13, 25)

err_qr = []
err_ne = []
err_svd = []

for kappa in kappas:
    U, _ = np.linalg.qr(np.random.randn(m, n))
    V, _ = np.linalg.qr(np.random.randn(n, n))
    s = np.logspace(0, -np.log10(kappa), n)
    A = U @ np.diag(s) @ V.T
    x_true = np.random.randn(n)
    b = A @ x_true  # small residual

    Q, R = np.linalg.qr(A)
    x_q = np.linalg.solve(R, Q.T @ b)
    try:
        x_n = np.linalg.solve(A.T @ A, A.T @ b)
    except np.linalg.LinAlgError:
        x_n = np.full(n, np.nan)
    x_s = np.linalg.lstsq(A, b, rcond=None)[0]

    err_qr.append(np.linalg.norm(x_q - x_true) / np.linalg.norm(x_true))
    err_ne.append(np.linalg.norm(x_n - x_true) / np.linalg.norm(x_true))
    err_svd.append(np.linalg.norm(x_s - x_true) / np.linalg.norm(x_true))

eps = np.finfo(float).eps

plt.figure(figsize=(9, 5.5))
plt.loglog(kappas, err_qr, 'bs-', markersize=4, label='Householder QR')
plt.loglog(kappas, err_ne, 'ro-', markersize=4, label='Normal equations')
plt.loglog(kappas, err_svd, 'g^-', markersize=4, label='SVD (lstsq)')
plt.loglog(kappas, kappas * eps, 'b--', alpha=0.4, label=r'$\kappa \cdot \epsilon$')
plt.loglog(kappas, kappas**2 * eps, 'r--', alpha=0.4, label=r'$\kappa^2 \cdot \epsilon$')
plt.axhline(1, color='gray', linestyle=':', alpha=0.5)
plt.xlabel(r'$\kappa(A)$')
plt.ylabel('Relative error')
plt.title('Least squares accuracy: QR vs Normal Equations vs SVD (small residual)')
plt.legend(fontsize=9)
plt.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

## The Role of the Residual: Small vs. Large $\theta$

In [ ]:
np.random.seed(7)
m, n = 80, 10
kappa_fixed = 1e6

U, _ = np.linalg.qr(np.random.randn(m, n))
V, _ = np.linalg.qr(np.random.randn(n, n))
s = np.logspace(0, -np.log10(kappa_fixed), n)
A = U @ np.diag(s) @ V.T

x_true = np.random.randn(n)

# Vary theta by adding noise orthogonal to the column space
Q_full, _ = np.linalg.qr(np.random.randn(m, m))
P_perp = np.eye(m) - U @ U.T  # projector onto complement of col(A)
noise_dir = P_perp @ Q_full[:, 0]  # a unit vector perp to col(A)
noise_dir = noise_dir / np.linalg.norm(noise_dir)

noise_scales = np.logspace(-10, 1, 20)
err_qr_theta = []
err_ne_theta = []
thetas = []

for scale in noise_scales:
    b = A @ x_true + scale * noise_dir
    theta = np.arcsin(np.linalg.norm(b - A @ np.linalg.lstsq(A, b, rcond=None)[0]) / np.linalg.norm(b))
    thetas.append(theta)

    Q, R = np.linalg.qr(A)
    x_q = np.linalg.solve(R, Q.T @ b)
    x_true_ls = np.linalg.lstsq(A, b, rcond=None)[0]  # "best" answer

    try:
        x_n = np.linalg.solve(A.T @ A, A.T @ b)
    except:
        x_n = np.full(n, np.nan)

    err_qr_theta.append(np.linalg.norm(x_q - x_true_ls) / np.linalg.norm(x_true_ls))
    err_ne_theta.append(np.linalg.norm(x_n - x_true_ls) / np.linalg.norm(x_true_ls))

plt.figure(figsize=(9, 5))
plt.loglog(thetas, err_qr_theta, 'bs-', markersize=4, label='Householder QR')
plt.loglog(thetas, err_ne_theta, 'ro-', markersize=4, label='Normal equations')
plt.xlabel(r'$\theta$ (angle between $b$ and column space)')
plt.ylabel('Relative error in $x$')
plt.title(f'Effect of residual size on accuracy ($\\kappa = {kappa_fixed:.0e}$)')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

## Normal Equations: Safe vs. Dangerous Zone

In [ ]:
np.random.seed(10)
m, n = 100, 10

print(f"{'kappa(A)':>12s}  {'kappa(A^TA)':>14s}  {'QR error':>12s}  {'NE error':>12s}  {'NE safe?':>10s}")
print('-' * 65)

for log_kap in [2, 4, 6, 8, 10, 12]:
    kappa = 10**log_kap
    U, _ = np.linalg.qr(np.random.randn(m, n))
    V, _ = np.linalg.qr(np.random.randn(n, n))
    s = np.logspace(0, -log_kap, n)
    A = U @ np.diag(s) @ V.T
    x_true = np.random.randn(n)
    b = A @ x_true

    Q, R = np.linalg.qr(A)
    x_qr = np.linalg.solve(R, Q.T @ b)
    try:
        x_ne = np.linalg.solve(A.T @ A, A.T @ b)
    except:
        x_ne = np.full(n, np.nan)

    e_qr = np.linalg.norm(x_qr - x_true) / np.linalg.norm(x_true)
    e_ne = np.linalg.norm(x_ne - x_true) / np.linalg.norm(x_true)
    safe = 'Yes' if kappa**2 * eps < 0.01 else 'NO'
    print(f"{kappa:12.0e}  {kappa**2:14.0e}  {e_qr:12.2e}  {e_ne:12.2e}  {safe:>10s}")

## The MGS Augmented Matrix Trick

Apply MGS to $[A \; b]$, then solve $R_{1:n,1:n} x = R_{1:n,n+1}$.  
This avoids using the non-orthogonal $Q$ explicitly.

In [ ]:
np.random.seed(3)
m, n = 80, 10
kappas = np.logspace(1, 13, 20)

err_mgs_direct = []
err_mgs_aug = []
err_hh = []

for kappa in kappas:
    U, _ = np.linalg.qr(np.random.randn(m, n))
    V, _ = np.linalg.qr(np.random.randn(n, n))
    s = np.logspace(0, -np.log10(kappa), n)
    A = U @ np.diag(s) @ V.T
    x_true = np.random.randn(n)
    b = A @ x_true

    # Householder
    Q, R = np.linalg.qr(A)
    x_h = np.linalg.solve(R, Q.T @ b)
    err_hh.append(np.linalg.norm(x_h - x_true) / np.linalg.norm(x_true))

    # MGS direct
    Q_m, R_m = modified_gs(A)
    x_md = np.linalg.solve(R_m, Q_m.T @ b)
    err_mgs_direct.append(np.linalg.norm(x_md - x_true) / np.linalg.norm(x_true))

    # MGS augmented
    Ab = np.column_stack([A, b])
    _, R_a = modified_gs(Ab)
    x_ma = np.linalg.solve(R_a[:n, :n], R_a[:n, n])
    err_mgs_aug.append(np.linalg.norm(x_ma - x_true) / np.linalg.norm(x_true))

plt.figure(figsize=(9, 5.5))
plt.loglog(kappas, err_hh, 'g^-', markersize=4, label='Householder QR')
plt.loglog(kappas, err_mgs_aug, 'bs-', markersize=4, label='MGS (augmented)')
plt.loglog(kappas, err_mgs_direct, 'ro-', markersize=4, label='MGS (direct $Q^Tb$)')
plt.loglog(kappas, kappas * eps, 'k--', alpha=0.4, label=r'$\kappa \cdot \epsilon$')
plt.axhline(1, color='gray', linestyle=':', alpha=0.5)
plt.xlabel(r'$\kappa(A)$')
plt.ylabel('Relative error')
plt.title('MGS direct vs augmented trick vs Householder')
plt.legend(fontsize=9)
plt.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

## Summary Visualization: The Three Methods

In [ ]:
methods = ['Householder QR', 'Normal Eqs', 'SVD']
stable = [True, False, True]
costs_label = [r'$2mn^2$', r'$mn^2 + n^3/3$', r'$2mn^2 + 11n^3$']
eff_kappa = [r'$\kappa$', r'$\kappa^2$', r'$\kappa$']

fig, ax = plt.subplots(figsize=(8, 3))
ax.axis('off')
table_data = [
    ['Method', 'Backward Stable?', 'Eff. $\\kappa$ (small residual)', 'Cost'],
    ['Householder QR', 'Yes', '$\\kappa$', '$2mn^2$'],
    ['Normal equations', 'NO', '$\\kappa^2$', '$mn^2 + n^3/3$'],
    ['SVD', 'Yes', '$\\kappa$', '$2mn^2 + 11n^3$'],
]

colors = [['lightblue']*4,
          ['lightgreen']*4,
          ['lightsalmon']*4,
          ['lightgreen']*4]

table = ax.table(cellText=table_data, cellColours=colors, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 1.8)
ax.set_title('Summary: Least Squares Methods Compared', fontsize=13, pad=20)
plt.tight_layout()
plt.show()